In [81]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [82]:
force_plate = pd.read_excel(r"C:\Users\ica-sc\Downloads\Inseason 24 CMJ.xlsx", sheet_name=None, engine="openpyxl")
print("Sheet names:", force_plate.keys())
baseline = force_plate["forcedecks-test-export-08_10_20"]
weekly = force_plate["Sheet1"]

print(baseline.head())
print(weekly.head())

Sheet names: dict_keys(['forcedecks-test-export-08_10_20', 'Sheet1'])
                Name       Date  RSI-modified [m/s]   \
0       Abe Del Real 2024-08-10                 0.83   
1       Abe Del Real 2024-07-31                 0.91   
2  Blake Antzaloutas 2024-08-10                 0.75   
3  Blake Antzaloutas 2024-07-31                 0.71   
4          Dave Main 2024-08-10                 0.72   

   Jump Height (Imp-Mom) [cm]  Concentric Impulse % (Asym) (%)  \
0                         49.9                          10.8 L   
1                         51.3                          10.6 L   
2                         48.4                           2.2 L   
3                         44.2                          11.9 R   
4                         41.4                           5.9 R   

  Eccentric Deceleration Impulse % (Asym) (%) Landing Impulse % (Asym) (%)  \
0                                      27.4 L                        2.6 R   
1                                      3

In [83]:
weekly.head()

,Name,Date,RSI-modified [m/s],Jump Height (Imp-Mom) [cm],Concentric Impulse % (Asym) (%),Eccentric Deceleration Impulse % (Asym) (%),Landing Impulse % (Asym) (%),Unnamed: 7,Name.1,Date.1,...,RSI-modified [m/s] .12,Name.13,Date.13,RSI-modified [m/s] .13,Name.14,Date.14,RSI-modified [m/s] .14,Avg,Today vs Avg,vs Last Week
0,Abe Del Real,2024-08-20,0.99,53.3,6.7 L,26.3 L,30.3 L,NaN,Abe Del Real,2024-08-29,...,1.06,Abe Del Real,2024-12-04,1.07,Abe Del Real,2024-12-11,1.03,1.092667,-0.060841,-0.038835
1,Blake Antzaloutas,2024-08-20,0.75,48.1,12.3 R,1 L,39.6 R,NaN,Blake Antzaloutas,2024-08-29,...,0.76,Blake Antzaloutas,2024-12-04,0.79,Blake Antzaloutas,2024-12-11,0.73,0.75,-0.027397,-0.082192
2,Blake Cotton,2024-08-20,0.90,48.7,5.3 L,15.2 R,20.9 R,NaN,Blake Cotton,2024-08-29,...,1.08,Blake Cotton,2024-12-04,1.06,Blake Cotton,2024-12-11,0.99,1.014667,-0.024916,-0.070707
3,Chaz Davis,2024-08-20,0.89,47.9,5.6 R,26.9 R,20.6 L,NaN,Chaz Davis,2024-08-29,...,1.17,Chaz Davis,2024-12-04,1.03,Chaz Davis,2024-12-11,1.07,1.053333,0.015576,0.037383
4,Christian Calhoun,2024-08-20,1.32,55.1,3.4 L,16.3 L,8.8 R,NaN,NaN,NaT,...,NaN,Christian Calhoun,2024-12-04,1.46,Christian Calhoun,2024-12-11,1.50,1.365455,0.089697,0.026667


In [84]:
print(weekly.columns)

Index(['Name', 'Date', 'RSI-modified [m/s] ', 'Jump Height (Imp-Mom) [cm] ',
       'Concentric Impulse % (Asym) (%)',
       'Eccentric Deceleration Impulse % (Asym) (%)',
       'Landing Impulse % (Asym) (%)', 'Unnamed: 7', 'Name.1', 'Date.1',
       'RSI-modified [m/s] .1', 'Jump Height (Imp-Mom) [cm] .1',
       'Concentric Impulse % (Asym) (%).1',
       'Eccentric Deceleration Impulse % (Asym) (%).1',
       'Landing Impulse % (Asym) (%).1', 'Unnamed: 15', 'Name.2', 'Date.2',
       'RSI-modified [m/s] .2', 'Jump Height (Imp-Mom) [cm] .2',
       'Concentric Impulse % (Asym) (%).2',
       'Eccentric Deceleration Impulse % (Asym) (%).2',
       'Landing Impulse % (Asym) (%).2', 'Name.3', 'Date.3',
       'RSI-modified [m/s] .3', 'Name.4', 'Date.4', 'RSI-modified [m/s] .4',
       'Name.5', 'Date.5', 'RSI-modified [m/s] .5', 'Name.6', 'Date.6',
       'RSI-modified [m/s] .6', 'Name.7', 'Date.7', 'RSI-modified [m/s] .7',
       'Name.8', 'Date.8', 'RSI-modified [m/s] .8', 'Name.9',

In [85]:
# Replace spaces with underscores for easier matching
weekly.columns = weekly.columns.str.replace(" ", "_")

# Drop unnamed columns dynamically
weekly = weekly.loc[:, ~weekly.columns.str.contains('^Unnamed')]

# Check column names for verification
print("Column Names in DataFrame:")
print(weekly.columns)

# Identify base column names (week 1 doesn't have a suffix, week 2+ have .1, .2, etc.)
base_columns = ['Name', 'Date', 'RSI-modified_[m/s]_', 'Jump_Height_(Imp-Mom)_[cm]_',
                'Concentric_Impulse_%_(Asym)_(%)',
                'Eccentric_Deceleration_Impulse_%_(Asym)_(%)',
                'Landing_Impulse_%_(Asym)_(%)']

num_weeks = 15  # Assuming there are 15 weeks of data

# Create individual DataFrames dynamically and store them in globals()
for i in range(num_weeks):
    if i == 0:
        week_columns = base_columns  # Week 1 has no suffix
    else:
        # Only pick columns with the correct suffix if they exist
        week_columns = [f"{col}.{i}" if f"{col}.{i}" in weekly.columns else None for col in base_columns]
        # Remove None values (columns that do not exist)
        week_columns = [col for col in week_columns if col is not None]
    
    # Extract the relevant columns for the week and create the DataFrame
    week_df = weekly[week_columns].copy()  # Extract relevant columns
    globals()[f"force_week_{i+1}"] = week_df  # Store in globals()
# Check if the DataFrames were created correctly
for i in range(num_weeks):
    try:
        # Get the DataFrame variable by name
        week_df = globals()[f"force_week_{i+1}"]
        print(f"force_week_{i+1} DataFrame created successfully!")
    except KeyError:
        print(f"force_week_{i+1} DataFrame does not exist!")

# Iterate through the dynamically created DataFrame variables and add the "Avg" column
for i in range(num_weeks):
        # Get the DataFrame variable by name
    week_df = globals()[f"force_week_{i+1}"]
        
        # Drop rows with NA values before calculation
    week_df = week_df.dropna()  # Drop rows with NA values
        
        # Add the 'Avg' column to the DataFrame
    week_df['Avg'] = weekly['Avg']  # Assuming the 'Avg' column value is the same for all rows
        
        # Add the 'RSI-modified' column for that specific week
    rsi_column = f"RSI-modified_[m/s]_.{i}" if i > 0 else "RSI-modified_[m/s]_"
        
        # Calculate the "Today vs Avg" column
    week_df['Today_vs_Avg'] = ((week_df[rsi_column] - week_df['Avg']) / week_df[rsi_column]) * 100

        # Update the variable with the new 'Avg' and 'Today_vs_Avg' columns
    globals()[f"force_week_{i+1}"] = week_df

Column Names in DataFrame:
Index(['Name', 'Date', 'RSI-modified_[m/s]_', 'Jump_Height_(Imp-Mom)_[cm]_',
       'Concentric_Impulse_%_(Asym)_(%)',
       'Eccentric_Deceleration_Impulse_%_(Asym)_(%)',
       'Landing_Impulse_%_(Asym)_(%)', 'Name.1', 'Date.1',
       'RSI-modified_[m/s]_.1', 'Jump_Height_(Imp-Mom)_[cm]_.1',
       'Concentric_Impulse_%_(Asym)_(%).1',
       'Eccentric_Deceleration_Impulse_%_(Asym)_(%).1',
       'Landing_Impulse_%_(Asym)_(%).1', 'Name.2', 'Date.2',
       'RSI-modified_[m/s]_.2', 'Jump_Height_(Imp-Mom)_[cm]_.2',
       'Concentric_Impulse_%_(Asym)_(%).2',
       'Eccentric_Deceleration_Impulse_%_(Asym)_(%).2',
       'Landing_Impulse_%_(Asym)_(%).2', 'Name.3', 'Date.3',
       'RSI-modified_[m/s]_.3', 'Name.4', 'Date.4', 'RSI-modified_[m/s]_.4',
       'Name.5', 'Date.5', 'RSI-modified_[m/s]_.5', 'Name.6', 'Date.6',
       'RSI-modified_[m/s]_.6', 'Name.7', 'Date.7', 'RSI-modified_[m/s]_.7',
       'Name.8', 'Date.8', 'RSI-modified_[m/s]_.8', 'Name.9', '

C:\Users\ica-sc\AppData\Local\Temp\ipykernel_7708\2968177806.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  week_df['Avg'] = weekly['Avg']  # Assuming the 'Avg' column value is the same for all rows
C:\Users\ica-sc\AppData\Local\Temp\ipykernel_7708\2968177806.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  week_df['Today_vs_Avg'] = ((week_df[rsi_column] - week_df['Avg']) / week_df[rsi_column]) * 100
C:\Users\ica-sc\AppData\Local\Temp\ipykernel_7708\2968177806.py:50: SettingWithCopyWarning: 
A val

In [87]:
force_week_10.head()

,Name.9,Date.9,RSI-modified_[m/s]_.9,Avg,Today_vs_Avg
0,Abe Del Real,2024-10-31,1.30,1.092667,15.948718
2,Blake Cotton,2024-10-31,0.97,1.014667,-4.604811
3,Chaz Davis,2024-10-31,1.12,1.053333,5.952381
4,Christian Calhoun,2024-10-31,1.32,1.365455,-3.443526
5,CJ Hutton,2024-10-31,0.93,0.904667,2.724014
